In [1]:
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

cmd = [sys.executable, str(ROOT / "scripts" / "03_build_features.py")]
r = subprocess.run(cmd, cwd=str(ROOT), capture_output=True, text=True)

print("returncode:", r.returncode)
print("stdout:\n", r.stdout[:4000])
print("stderr:\n", r.stderr[:2000])

returncode: 0
stdout:
 [fineas] wrote: C:\Users\quantbase\Desktop\fineas\data\meta\runs\run=20260304T102000Z\features_report.json
[fineas] summary: ok_parts=96/96, qc_failed=0, error=0, missing_norm=0

stderr:
 


In [2]:
# Load features_report.json and inspect summary

import json

runs_dir = ROOT / "data" / "meta" / "runs"
latest = sorted([p for p in runs_dir.glob("run=*") if p.is_dir()])[-1]
report_path = latest / "features_report.json"
print("report:", report_path)

payload = json.loads(report_path.read_text(encoding="utf-8"))
summary = payload["summary"]
summary

report: C:\Users\quantbase\Desktop\fineas\data\meta\runs\run=20260304T102000Z\features_report.json


{'run_id': 'run=20260304T102000Z',
 'created_utc': '2026-03-05T05:45:30.481963+00:00',
 'interval': '1d',
 'start': '2022-01-01',
 'end': '2024-01-01',
 'symbols': ['AAPL', 'MSFT', 'NVDA', 'NEWGEN.NS'],
 'months': [[2022, 1],
  [2022, 2],
  [2022, 3],
  [2022, 4],
  [2022, 5],
  [2022, 6],
  [2022, 7],
  [2022, 8],
  [2022, 9],
  [2022, 10],
  [2022, 11],
  [2022, 12],
  [2023, 1],
  [2023, 2],
  [2023, 3],
  [2023, 4],
  [2023, 5],
  [2023, 6],
  [2023, 7],
  [2023, 8],
  [2023, 9],
  [2023, 10],
  [2023, 11],
  [2023, 12]],
 'lookback_rows': 20,
 'expected_parts': 96,
 'ok_parts': 96,
 'missing_norm_parts': 0,
 'empty_norm_parts': 0,
 'missing_cols_norm_parts': 0,
 'qc_failed_parts': 0,
 'error_parts': 0,
 'feature_cols': ['log_ret_1',
  'ret_2',
  'ret_5',
  'range_pct',
  'body_pct',
  'close_pos_in_range',
  'volume_chg_1',
  'volatility_5',
  'sma_5_ratio',
  'sma_10_ratio'],
 'feature_warn_partitions': 4,
 'prev_tail_used_parts': 92,
 'feature_nan_counts_by_symbol': {'AAPL': {'l

In [4]:
# acceptance assertions (report-level)

assert summary["expected_parts"] == summary["ok_parts"], summary
assert summary["missing_norm_parts"] == 0, summary
assert summary["empty_norm_parts"] == 0, summary
assert summary["missing_cols_norm_parts"] == 0, summary
assert summary["qc_failed_parts"] == 0, summary
assert summary["error_parts"] == 0, summary
assert len(payload["results"]) == summary["expected_parts"], {
    "results_len": len(payload["results"]),
    "expected_parts": summary["expected_parts"],
}

print("Batch runner (full universe) REPORT CHECK: PASS")

Batch runner (full universe) REPORT CHECK: PASS


In [6]:
# Sample one features partition and recheck QC

import pandas as pd

from fineas.features.technicals import V1_FEATURE_COLS
from fineas.io.qc import qc_features_partition

ok_items = [x for x in payload["results"] if x.get("status") == "ok"]
sample = ok_items[0]
p = Path(sample["out_path"])
df = pd.read_parquet(p)

print(sample["symbol"], sample["year"], sample["month"], p)
print(df.shape)
df.head(8)

AAPL 2022 1 C:\Users\quantbase\Desktop\fineas\data\norm\features_daily\interval=1d\symbol=AAPL\year=2022\month=01\part-2022-01.parquet
(20, 20)


,ts,symbol,open,high,low,close,adj_close,volume,returns,movement,log_ret_1,ret_2,ret_5,range_pct,body_pct,close_pos_in_range,volume_chg_1,volatility_5,sma_5_ratio,sma_10_ratio
0,2022-01-03 00:00:00+00:00,AAPL,177.830002,182.880005,177.710007,182.009995,178.103653,104487900,NaN,NaN,NaN,NaN,NaN,2.840502,2.350555,0.831719,NaN,NaN,NaN,NaN
1,2022-01-04 00:00:00+00:00,AAPL,182.630005,182.940002,179.119995,179.699997,175.843246,99310400,-1.269152,fall,-0.012773,NaN,NaN,2.125769,-1.604341,0.151833,-4.955119,NaN,NaN,NaN
2,2022-01-05 00:00:00+00:00,AAPL,179.610001,180.169998,174.639999,174.919998,171.165817,94537600,-2.659999,fall,-0.026960,-3.895392,NaN,3.161445,-2.611215,0.050633,-4.805942,NaN,NaN,NaN
3,2022-01-06 00:00:00+00:00,AAPL,172.699997,175.300003,171.639999,172.000000,168.308533,96904000,-1.669308,fall,-0.016834,-4.284904,NaN,2.127909,-0.405325,0.098361,2.503131,NaN,NaN,NaN
4,2022-01-07 00:00:00+00:00,AAPL,172.889999,174.139999,171.029999,172.169998,168.474854,86709100,0.098819,freeze,0.000988,-1.572139,NaN,1.806355,-0.416450,0.366559,-10.520618,NaN,0.977350,NaN
5,2022-01-10 00:00:00+00:00,AAPL,169.080002,172.500000,168.169998,172.190002,168.494431,106765600,0.011620,freeze,0.000116,0.110451,-5.395298,2.514665,1.839366,0.928407,23.130790,1.168242,0.988484,NaN
6,2022-01-11 00:00:00+00:00,AAPL,172.320007,175.179993,170.820007,175.080002,171.322418,76138300,1.678386,rise,0.016645,1.690202,-2.570942,2.490282,1.601668,0.977066,-28.686487,1.688065,1.010435,NaN
7,2022-01-12 00:00:00+00:00,AAPL,176.119995,177.179993,174.820007,175.529999,171.762756,74805200,0.257023,freeze,0.002567,1.939723,0.348749,1.344491,-0.334997,0.300846,-1.750893,1.188610,1.012319,NaN


In [7]:
qc = qc_features_partition(
    df,
    name=f"recheck:{p.name}",
    expected_feature_cols=V1_FEATURE_COLS,
)
qc.to_dict()

{'ok': True,
 'hard_fails': [],
 'warnings': ['recheck:part-2022-01.parquet: NaNs present in feature columns (warmup likely) count=7'],
 'stats': {'name': 'recheck:part-2022-01.parquet',
  'rows': 20,
  'key_name': 'recheck:part-2022-01.parquet/key',
  'key_rows': 20,
  'key_key_cols': ['symbol', 'ts'],
  'key_dup_keys_n': 0,
  'time_name': 'recheck:part-2022-01.parquet/time',
  'time_monotonic_bad_symbols': [],
  'time_max_gap_days': 4.0,
  'time_gap_warn_symbols': [],
  'movement_invalid_values': [],
  'movement_present_where_returns_nan_n': 0,
  'feature_cols': ['log_ret_1',
   'ret_2',
   'ret_5',
   'range_pct',
   'body_pct',
   'close_pos_in_range',
   'volume_chg_1',
   'volatility_5',
   'sma_5_ratio',
   'sma_10_ratio'],
  'feature_cols_n': 10,
  'feature_non_numeric_cols': [],
  'feature_inf_counts': {'log_ret_1': 0,
   'ret_2': 0,
   'ret_5': 0,
   'range_pct': 0,
   'body_pct': 0,
   'close_pos_in_range': 0,
   'volume_chg_1': 0,
   'volatility_5': 0,
   'sma_5_ratio': 0,


In [9]:
# Quick diagnostics: status counts + warmup NaN summary table

from collections import Counter

status_counts = Counter([x.get("status") for x in payload["results"]])
print("status_counts:", status_counts)

nan_tbl = pd.DataFrame(summary["feature_nan_counts_by_symbol"]).T
nan_tbl

status_counts: Counter({'ok': 96})


,log_ret_1,ret_2,ret_5,range_pct,body_pct,close_pos_in_range,volume_chg_1,volatility_5,sma_5_ratio,sma_10_ratio
AAPL,1,2,5,0,0,0,1,5,4,9
MSFT,1,2,5,0,0,0,1,5,4,9
NVDA,1,2,5,0,0,0,1,5,4,9
NEWGEN.NS,1,2,5,0,0,0,1,5,4,9
